### RUN

In [ ]:

from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import frameitRunner

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import math

conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_batsirai.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_belna.yml"))

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_chido.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_ianos.yml"))

conf.printID()

runner = frameitRunner(conf)
runner.run()

In [ ]:
runner.ds_user['level'].UT

### PLOT TRAJ

In [ ]:
proj = ccrs.PlateCarree()
atm_model = runner.conf.atm_model
tracking_method = runner.conf.tracking_method
# ----------------------------------------------------------------------
# Choix du groupe vertical / dimension verticale
# ----------------------------------------------------------------------
if atm_model == "AROME":
    vert_group = "heightAboveGround"
    vert_dim   = "heightAboveGround"
    k_level    = 0
    uvel='u'
elif atm_model == "MNH":
    vert_group = "level"    # Ã  adapter si besoin pour MNH
    vert_dim   = "level"
    k_level    = 0
    uvel='UT'
else:
    raise ValueError(f"Modèle atmosphérique non géré: {runner.conf.atm_model!r}")

# DataArray u(time, z, lat, lon) ou (time, z, y, x)
u_all = runner.ds_user[vert_group][uvel].isel({vert_dim: k_level})
times = u_all["time"]
nt = times.size 

# Domaine lat/lon brut du modèle (pour clipper le zoom)
u0 = u_all.isel(time=0)
if atm_model == "AROME":
    data_lats = u0["latitude"].values
    data_lons = u0["longitude"].values
else:
    data_lats = runner.ds_user[vert_group]["latitude"].values
    data_lons = runner.ds_user[vert_group]["longitude"].values

data_lon_min = float(np.min(data_lons))
data_lon_max = float(np.max(data_lons))
data_lat_min = float(np.min(data_lats))
data_lat_max = float(np.max(data_lats))

margin = 0.75  # marge en degrés autour de la trackectoire ou du centre
if tracking_method == "fixed_box " : margin = 4.0  # marge en degrés autour de la trackectoire ou du centre


# --------------------------------
# Trackectoire prescrite
# --------------------------------
if tracking_method == "prescribed_track":
    trk_lat_aligned = trk_lon_aligned = None
    trk_lat_full = trk_lon_full = None
    trk = xr.open_dataset(runner.conf.prescribed_track_file)

    # on s'aligne sur les temps du modèle (hypothèse: exact match)
    trk_aligned = trk.sel(time=times)
    trk_lat_aligned = trk_aligned["latitude"].values
    trk_lon_aligned = trk_aligned["longitude"].values

    # pour tracer la trackectoire complète (ligne noire)
    track_lat = trk["latitude"].values
    track_lon = trk["longitude"].values

# --------------------------------
# Tracking method used
# --------------------------------
if tracking_method not in ["prescribed_track","fixed_box"]:
    if atm_model == "MNH":
        track_lat = data_lats[runner.track.cy.values,runner.track.cy.values]
        track_lon = data_lons[runner.track.cy.values,runner.track.cx.values]
    else: 
        track_lat = data_lats[runner.track.cy.values]
        track_lon = data_lons[runner.track.cx.values]

# print(track_lat,track_lon)

# --------------------------------
# Définition d'un zoom fixe
# --------------------------------

if tracking_method != "fixed_box":
    lon_min = float(np.min(track_lon)) - margin
    lon_max = float(np.max(track_lon)) + margin
    lat_min = float(np.min(track_lat)) - margin
    lat_max = float(np.max(track_lat)) + margin
    
else:
    # centre fixe défini dans la config
    lat0, lon0 = runner.conf.fix_subdomain_center
    lon_min = max(lon0 - margin, data_lon_min)
    lon_max = min(lon0 + margin, data_lon_max)
    lat_min = max(lat0 - margin, data_lat_min)
    lat_max = min(lat0 + margin, data_lat_max)


# on clippe au domaine du modèle
lon_min = max(lon_min, data_lon_min)
lon_max = min(lon_max, data_lon_max)
lat_min = max(lat_min, data_lat_min)
lat_max = min(lat_max, data_lat_max)

zoom_extent = [lon_min, lon_max, lat_min, lat_max]

# ----------------------------------------------------------------------
# 1) Figure de contrôle : un champ u
# ----------------------------------------------------------------------
# t_ref = 1
# u_ref = u_all.isel(time=t_ref)
# if tracking_method == 'fixed_box' :
#     x_center = lon0
#     y_center = lat0
# else: 
#     x_center = track_lon[t_ref]
#     y_center = track_lat[t_ref]

# fig, ax = plt.subplots(
#     1, 1,
#     subplot_kw={"projection": proj},
#     figsize=(6, 4),
# )

# ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
# ax.add_feature(cfeature.BORDERS, linewidth=0.5)
# ax.gridlines(draw_labels=True, linewidth=0.5, linestyle=":")

# if runner.conf.atm_model == "AROME":
#     lats_1d = u_ref["latitude"].values
#     lons_1d = u_ref["longitude"].values
#     lon2, lat2 = np.meshgrid(lons_1d, lats_1d)
# else:
#     lat2 = runner.ds_user[vert_group]["latitude"].values
#     lon2 = runner.ds_user[vert_group]["longitude"].values

# ax.set_extent(zoom_extent, crs=proj)

# pm = ax.pcolormesh(lon2, lat2, u_ref.values, transform=proj, shading="auto")
# fig.colorbar(pm, ax=ax, orientation="vertical", shrink=0.8)

# ax.set_title(
#     f"u ({vert_dim}={k_level}), time={np.datetime_as_string(u_ref['time'].values)}"
# )

# # pour visualiser aussi la trackectoire entière
# ax.plot(track_lon,track_lat,
#     color="k",linewidth=2.0,transform=proj,
# )
# ax.scatter(x_center,y_center,color="r",transform=proj,
# )

# ----------------------------------------------------------------------
# 2) Figure avec subplots pour chaque pas de temps
# ----------------------------------------------------------------------
ncols = 2
nrows = math.ceil(nt / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    subplot_kw={"projection": proj},
    figsize=(5 * ncols, 3 * nrows),
)
axes = np.atleast_1d(axes).ravel()

cx = runner.track["cx"]
cy = runner.track["cy"]

for tt in range(nt-1):
    ax = axes[tt]
    u_t = u_all.isel(time=tt)

    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.5, linestyle=":")

    if runner.conf.atm_model == "AROME":
        lats_1d = u_t["latitude"].values
        lons_1d = u_t["longitude"].values
        lon2, lat2 = np.meshgrid(lons_1d, lats_1d)
    else:
        lat2 = runner.ds_user[vert_group]["latitude"].values
        lon2 = runner.ds_user[vert_group]["longitude"].values

    ax.set_extent(zoom_extent, crs=proj)

    pm = ax.pcolormesh(lon2, lat2, u_t.values, transform=proj, shading="auto")
    fig.colorbar(pm, ax=ax, shrink=0.81, pad=0.14, aspect=20,label='zonal wind')
    

    # tracer la trackectoire complète en noir
    ax.plot(
        track_lon,
        track_lat,
        color="k",
        linewidth=2.0,
        transform=proj,
    )
    
    
        # indices de la boîte domainpée
    ixmin = int(runner.track["ix_min_domain"].values[tt])
    ixmax = int(runner.track["ix_max_domain"].values[tt])
    iymin = int(runner.track["iy_min_domain"].values[tt])
    iymax = int(runner.track["iy_max_domain"].values[tt])

    # coins de la boîte (ordre fermé)
    xs = [ixmin, ixmax, ixmax, ixmin, ixmin]
    ys = [iymin, iymin, iymax, iymax, iymin]
    if atm_model == 'MNH':
        box_lon = data_lons[ys, xs]
        box_lat = data_lats[ys, xs]
    else:
        box_lon = data_lons[xs]
        box_lat = data_lats[ys]
        

    # tracé de la boîte en lon/lat
    ax.plot(
        box_lon, box_lat, "-r",
        transform=ccrs.PlateCarree()
    )

    # Indices du centre au pas de temps courant
    if "time" in cx.dims:
        i = int(cx.isel(time=tt).values)
        j = int(cy.isel(time=tt).values)
    else:
        i = int(cx.values)
        j = int(cy.values)

    if runner.conf.atm_model == "AROME":
        lon_c = float(lons_1d[i])
        lat_c = float(lats_1d[j])
    else:
        lon_c = float(lon2[j, i])
        lat_c = float(lat2[j, i])

    ax.scatter(
        lon_c,
        lat_c,
        marker="x",
        s=75, color='r',
        transform=proj,
        label="Extracted box center",
    )

    # point de tracjectoire prescrite/tracké
    ax.scatter(
        track_lon,
        track_lat,
        marker="o",
        s=75,
        facecolors="none",
        edgecolors="k",
        transform=proj,
        label="Tracking center",
    )
    
    # tracé des bords du domaine (optionnel)
    ax.plot(
        [data_lons.min(), data_lons.min(), data_lons.max(), data_lons.max(), data_lons.min()],
        [data_lats.min(), data_lats.max(), data_lats.max(), data_lats.min(), data_lats.min()],
        "--k",
        lw=2,
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(f'{runner.track["time"][tt].values}')
    ax.set_title(f"{runner.conf.simulation_name} ; {np.datetime_as_string(times[tt].values)[0:13]}",fontweight='bold')
    ax.legend(loc="upper left", fontsize=8)

# masquer les panneaux inutilisés
for k in range(nt-1, nrows * ncols):
    fig.delaxes(axes[k])

# fig.suptitle("Centres trackés par pas de temps", y=1.01)
plt.tight_layout()

In [ ]:
np.datetime_as_string(times[tt].values)[0:13]

### TEST MNH

In [ ]:

var='UM10';  vert_coord = 'surface' ; lev = 2

for tt in range(0,np.size(runner.ds_user[vert_coord][var]['time'])):
    fig,ax = plt.subplots(ncols=2,figsize=(10,5))
    # runner.ds_user[vert_coord][var].isel(time=tt,level=lev).plot.contourf(ax=ax[0])
    # runner.ds_box[vert_coord][var].isel(time=tt,level=lev).plot.contourf(ax=ax[1])
    runner.ds_user[vert_coord][var].isel(time=tt).plot.contourf(ax=ax[0])
    runner.ds_box[vert_coord][var].isel(time=tt).plot.contourf(ax=ax[1])

In [ ]:
runner.ds_box['level'].attrs

### TEST AROME

In [ ]:
var='prmsl';  vert_coord = 'surface' ; lev = 2

for tt in range(0,np.size(runner.ds_user[vert_coord][var]['time'])):
    fig,ax = plt.subplots(ncols=2,figsize=(10,5))
    # runner.ds_user[vert_coord][var].isel(time=tt,level=lev).plot.contourf(ax=ax[0])
    # runner.ds_box[vert_coord][var].isel(time=tt,level=lev).plot.contourf(ax=ax[1])
    runner.ds_user[vert_coord][var].isel(time=tt).plot.contourf(ax=ax[0])
    runner.ds_box[vert_coord][var].isel(time=tt).plot.contourf(ax=ax[1])